In [2]:
import polars as pl

# 1. Load the raw dataset
# (Notice the '../' because we are stepping out of the notebooks folder and into the data folder)
data_path = "../data/raw/raw_binding_data.csv"
df = pl.read_csv(data_path)

# 2. Print the total size
print(f"Total rows and columns: {df.shape}")

# 3. Display the first 5 rows to see the raw strings
# (In a notebook, just putting 'df.head()' at the end prints a beautiful HTML table)
df.head()

Total rows and columns: (52274, 5)


Drug_ID,Drug,Target_ID,Target,Y
f64,str,str,str,f64
444607.0,"""Cc1ccc(CNS(=O)(=O)c2ccc(S(N)(=…","""P00918""","""MSHHWGYGKHNGPEHWHKDFPIAKGERQSP…",0.46
4316.0,"""COc1ccc(CNS(=O)(=O)c2ccc(S(N)(…","""P00918""","""MSHHWGYGKHNGPEHWHKDFPIAKGERQSP…",0.49
4293.0,"""NS(=O)(=O)c1ccc(S(=O)(=O)NCc2c…","""P00918""","""MSHHWGYGKHNGPEHWHKDFPIAKGERQSP…",0.83
1611.0,"""NS(=O)(=O)c1cc2c(s1)S(=O)(=O)N…","""P00918""","""MSHHWGYGKHNGPEHWHKDFPIAKGERQSP…",0.2
1612.0,"""COc1ccc(N2CC(O)c3cc(S(N)(=O)=O…","""P00918""","""MSHHWGYGKHNGPEHWHKDFPIAKGERQSP…",0.16


In [3]:
# 1. Add two new columns that count the characters in the strings
df_lengths = df.with_columns([
    pl.col("Drug").str.len_chars().alias("Drug_Length"),
    pl.col("Target").str.len_chars().alias("Target_Length")
])

# 2. Print the statistical summary (min, max, average lengths)
print("--- Molecular Sequence Length Statistics ---")
df_lengths.select(["Drug_Length", "Target_Length"]).describe()

--- Molecular Sequence Length Statistics ---


statistic,Drug_Length,Target_Length
str,f64,f64
"""count""",52274.0,52274.0
"""null_count""",0.0,0.0
"""mean""",63.204863,703.202548
"""std""",44.25612,391.703767
"""min""",8.0,53.0
"""25%""",46.0,426.0
"""50%""",56.0,596.0
"""75%""",66.0,942.0
"""max""",973.0,4128.0


In [4]:
# 1. Define our hardware safety limits
MAX_TARGET_LEN = 1000
MAX_DRUG_LEN = 200

print(f"Original dataset size: {df.shape[0]}")

# 2. Filter the dataset: Keep only rows where BOTH conditions are true
df_filtered = df_lengths.filter(
    (pl.col("Target_Length") <= MAX_TARGET_LEN) & 
    (pl.col("Drug_Length") <= MAX_DRUG_LEN)
)

print(f"Filtered dataset size: {df_filtered.shape[0]}")

# 3. Calculate how much data survived the cut
survival_rate = (df_filtered.shape[0] / df.shape[0]) * 100
print(f"Survival Rate: {survival_rate:.2f}%")

Original dataset size: 52274
Filtered dataset size: 41109
Survival Rate: 78.64%


In [6]:
import os
import polars as pl

# --- NEW CRITICAL STEP: PRECOMPUTE pKd ---
# Use the correct column name "Y" for the raw Kd values
df_filtered = df_filtered.with_columns(
    (9.0 - (pl.col("Y") + 1e-10).log10()).cast(pl.Float32).alias("pkd")
)

# 1. THE FINAL EXAM (Cold-Target Split for Disease X)
unique_targets = df_filtered.select("Target").unique()
test_targets = unique_targets.sample(fraction=0.1, seed=42)
test_target_list = test_targets.get_column("Target").to_list()

df_test = df_filtered.filter(pl.col("Target").is_in(test_target_list))

# 2. THE REMAINING DATA
# Everything that isn't in the test set
df_remaining = df_filtered.filter(~pl.col("Target").is_in(test_target_list))

# 3. THE PRACTICE QUIZZES & TEXTBOOKS (Train/Val Split)
# Shuffle the remaining data completely to ensure random distribution
df_remaining_shuffled = df_remaining.sample(fraction=1.0, shuffle=True, seed=42)

# Calculate exactly 15% for validation
val_size = int(df_remaining_shuffled.shape[0] * 0.15)

# Slice the dataframe cleanly
df_val = df_remaining_shuffled.head(val_size)
df_train = df_remaining_shuffled.tail(df_remaining_shuffled.shape[0] - val_size)

print("--- The Cortex Data Trinity ---")
print(f"Training Pairs (85% of remaining):   {df_train.shape[0]}")
print(f"Validation Pairs (15% of remaining): {df_val.shape[0]}")
print(f"Zero-Shot Test Pairs (Disease X):    {df_test.shape[0]}")

# 4. PROFESSIONAL SERIALIZATION (Parquet)
# Create the processed directory
processed_dir = "../data/processed"
os.makedirs(processed_dir, exist_ok=True)

# Write to the highly optimized Parquet format (THIS OVERWRITES THE OLD FILES)
df_train.write_parquet(os.path.join(processed_dir, "train.parquet"))
df_val.write_parquet(os.path.join(processed_dir, "val.parquet"))
df_test.write_parquet(os.path.join(processed_dir, "test.parquet"))

print(f"\n[SUCCESS] Highly optimized .parquet files saved to {processed_dir}/")

--- The Cortex Data Trinity ---
Training Pairs (85% of remaining):   31371
Validation Pairs (15% of remaining): 5536
Zero-Shot Test Pairs (Disease X):    4202

[SUCCESS] Highly optimized .parquet files saved to ../data/processed/
